# mvp_model: train on Kaggle (GPU)
Run the cells from top to bottom. Only **cell 1** needs editing: which versions to train in this run.
Works on **Kaggle** and on **Google Colab**. Kaggle: right panel **Session options -> Accelerator -> GPU T4 x2**, and **Add Input** -> your dataset with the mvp_model code and `mvp_v1_packed.npz`. Colab: Runtime -> Change runtime type -> T4 GPU, upload the two files (see the guide).
Full guide: `docs/kaggle-guide.md`.

In [ ]:
# ---- cell 1: settings (the only cell you edit) ----
RUN_NAME = 'main'                                           # name of this run, used for the results zip
VARIANTS = ['full', 'hist', 'full_direct', 'hist_direct']   # run 1 (main). For run 2 use: ['noocc', 'concat']
SEEDS    = '0,1,2'                                          # 3 seeds
EPOCHS   = 60                                               # at most 60 epochs per training (early stopping is on)
PATIENCE = 10

In [ ]:
# ---- cell 2: find the code and the data, set up the working folder (Kaggle or Colab) ----
import glob, os, shutil, zipfile
ON_KAGGLE = os.path.exists('/kaggle/input')
WORK = '/kaggle/working' if ON_KAGGLE else '/content'
ROOTS = ['/kaggle/input'] if ON_KAGGLE else ['/content', '/content/drive/MyDrive']
def find(pattern):
    for r in ROOTS:
        hits = [h for h in glob.glob(f'{r}/**/{pattern}', recursive=True) if '/mvp_model/mvp_model/' not in h]
        if hits:
            return hits[0]
    return None
train_py = find('train.py')
if train_py is None:                                        # the code was uploaded as a zip that was not unpacked
    z = find('mvp_model*.zip')
    assert z, 'No train.py and no mvp_model zip found. Did you add the dataset as Input / upload the files?'
    zipfile.ZipFile(z).extractall(WORK + '/unzipped')
    train_py = glob.glob(WORK + '/unzipped/**/train.py', recursive=True)[0]
SRC = os.path.dirname(train_py)
CODE = WORK + '/mvp_model'
if os.path.abspath(SRC) != os.path.abspath(CODE):
    shutil.copytree(SRC, CODE, dirs_exist_ok=True, ignore=shutil.ignore_patterns('runs', '__pycache__', 'reports', 'packed'))
DATA = find('*_packed.npz')
assert DATA, 'mvp_v1_packed.npz not found. Upload it together with the code (see the guide).'
os.chdir(CODE)
print('platform:', 'Kaggle' if ON_KAGGLE else 'Colab / other')
print('code    :', CODE, '(from', SRC + ')')
print('data    :', DATA, f'({os.path.getsize(DATA) / 1e6:.1f} MB)')

In [ ]:
# ---- cell 3: is there a GPU? ----
import torch
print('torch', torch.__version__, '| GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    x = torch.randn(1000, 1000, device='cuda'); print('GPU test ok:', float((x @ x).sum()) != 0)
else:
    print('NO GPU. Right panel -> Session options -> Accelerator -> choose a GPU, then Run All again.')

In [ ]:
# ---- cell 4: check the networks (must end with TORCH CHECK PASSED) ----
!python tests/test_torch.py

In [ ]:
# ---- cell 5: smoke test, about 1 minute: one fold, 2 epochs (written to runs/smoke, never mixed with real results) ----
!python train.py "{DATA}" --variant full --seeds 0 --epochs 2 --fold 0

In [ ]:
# ---- cell 6: the real training (this is the long cell) ----
# A seed that is already finished is skipped, so after an interruption just run this cell again.
import time
os.makedirs('runs/logs', exist_ok=True)
for v in VARIANTS:
    t0 = time.time()
    print('=' * 30, v, '=' * 30)
    get_ipython().system(f'python train.py "{DATA}" --variant {v} --seeds {SEEDS} --epochs {EPOCHS} --patience {PATIENCE} 2>&1 | tee -a runs/logs/{v}.log')
    print(f'>>> {v} finished in {(time.time() - t0) / 60:.1f} min')

In [ ]:
# ---- cell 7: evaluation on Kaggle (optional: you can also run evaluate.py on your laptop later) ----
def files(v, suffix=''):
    return ','.join(sorted(glob.glob(f'runs/oof_{v}_s?{suffix}.npz')))

os.makedirs('reports', exist_ok=True)
for test, ref in (('full', 'hist'), ('full_direct', 'hist_direct')):
    if not (files(test) and files(ref)):
        continue
    cmd = f'python evaluate.py "{DATA}" --test {test} --ref {ref} --preds {test}={files(test)} --preds {ref}={files(ref)}'
    if files(test, '_shufscn'):
        cmd += f' --preds {test}_shufscn={files(test, "_shufscn")} --shuf-scn {test}_shufscn'
    for extra, flag in (('noocc', '--noocc'), ('concat', '--concat')):
        name = extra + ('_direct' if test.endswith('_direct') else '')
        if files(name):
            cmd += f' --preds {name}={files(name)} {flag} {name}'
    print('=' * 100, '\n', test, 'vs', ref, '\n', cmd)
    get_ipython().system(cmd + f' --out reports/eval_{test}.txt')

In [ ]:
# ---- cell 8: pack the results into ONE zip to download ----
import zipfile
zname = f'{WORK}/results_{RUN_NAME}.zip'
n = 0
with zipfile.ZipFile(zname, 'w', zipfile.ZIP_DEFLATED) as z:
    for pat in ('runs/oof_*.npz', 'runs/models/*.pt', 'runs/logs/*.log', 'reports/*.txt'):
        for f in sorted(glob.glob(pat)):
            z.write(f); n += 1
print(f'{n} files -> {zname} ({os.path.getsize(zname) / 1e6:.1f} MB)')
if ON_KAGGLE:
    print('Download it: notebook page -> Output tab -> results_*.zip   (after "Save Version" if you ran with Save & Run All)')
else:
    from google.colab import files
    files.download(zname)